# Run validation

This notebook run and generate validation reports for predictive ADMET models including 10 classification models and 1 regression model

In [ ]:
import subprocess
import pandas as pd


# For classification model
# Run validation
for prop in ['BBB', 'AMES', 'CYP1A2', 'CYP2C9', 'CYP2C19', 'CYP2D6', 'CYP3A4', 'hERG', 'PAMPA', 'PGP']:
    data = f'train_val_data/{prop}_test.csv'
    model = f'../../models/{prop}_clf_trained.jar'
    output = f"validations/{prop}_validation.txt"

    subprocess.run([
        "java", "-jar", "../../models/CPSign/cpsign-2.0.0-fatjar.jar", "validate",
        "--validation-property", "target",
        "-p", "CSV", data,
        "-cp", "0.5:1.0:0.01",
        "-m", model,
        "-ro", output
    ])


# Parse results
def parse_cpsign_file(filepath):
    with open(filepath) as f:
        lines = f.readlines()

    start = None
    for i, line in enumerate(lines):
        if line.startswith("Confidence"):
            start = i
            break

    return pd.read_csv(filepath, sep="\t", skiprows=start)


datasets = ['BBB', 'AMES', 'CYP1A2', 'CYP2C9', 'CYP2C19',
            'CYP2D6', 'CYP3A4', 'hERG', 'PAMPA', 'PGP']

dfs = []

for name in datasets:
    filepath = f"validations/{name}_validation.txt"

    df = parse_cpsign_file(filepath)

    df_08 = df[df["Confidence"] == 0.8].copy()
    df_08["Model"] = name

    dfs.append(df_08)

# Combine everything
final_df = pd.concat(dfs, ignore_index=True)

final_df.to_csv('validations/validations_clf.csv', index = False)

final_df

In [31]:
# For regression model
prop = 'Solubility'
data = f'train_val_data/{prop}_test.csv'
model = f'../../models/{prop}_rgs_trained.jar'
output = f"validations/{prop}_validation.txt"

subprocess.run([
    "java", "-jar", "../../models/CPSign/cpsign-2.0.0-fatjar.jar", "validate",
    "--validation-property", "target",
    "-p", "CSV", data,
    "-cp", "0.5:1.0:0.01",
    "-m", model,
    "-ro", output
])


                            -= CPSign - VALIDATE =-

Validating arguments... [done]
Loading model... [done]
Computing predictions... 
 - Processed 300/1997 molecules
 - Processed 600/1997 molecules
 - Processed 900/1997 molecules
 - Processed 1200/1997 molecules
 - Processed 1500/1997 molecules
 - Processed 1800/1997 molecules

Successfully predicted 1943 molecules. Failed predicting 54 records.

Writing overall statistics to file:
/Users/dinhu955/Desktop/RepurAgent/repuragent-web/analysis/CPSIgn_eval/validations/Solubility_validation.txt ... [done]



CompletedProcess(args=['java', '-jar', '../../models/CPSign/cpsign-2.0.0-fatjar.jar', 'validate', '--validation-property', 'target', '-p', 'CSV', 'train_val_data/Solubility_test.csv', '-cp', '0.5:1.0:0.01', '-m', '../../models/Solubility_rgs_trained.jar', '-ro', 'validations/Solubility_validation.txt'], returncode=0)